# Agentic Self-Instruct

Generated examples are worthless unless they separate a strong model from a weak one.

*A test everyone passes and a test nobody passes are the same test: neither tells you who
knows the material. The gap between two solvers IS the signal.*

## 0. Setup

`cp ../.env.example ../.env`, add `DEEPINFRA_API_KEY`. Four roles, three models — the judge is a
different family from both solvers, so it isn't grading its own homework.

In [ ]:
import json
import os
import random
import re
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

from dotenv import find_dotenv, load_dotenv
from openai import APIError, OpenAI, RateLimitError

load_dotenv(find_dotenv(usecwd=True))
client = OpenAI(
    api_key=os.environ["DEEPINFRA_API_KEY"],
    base_url="https://api.deepinfra.com/v1/openai",
)

CHALLENGER = "Qwen/Qwen3-235B-A22B-Instruct-2507"
STRONG     = "Qwen/Qwen3-235B-A22B-Instruct-2507"
WEAK       = "meta-llama/Meta-Llama-3.1-70B-Instruct-Turbo"
JUDGE      = "Qwen/Qwen2.5-72B-Instruct"   # different family: decorrelate from both solvers

SOURCE = open("source.md").read()   # fictional tariff: no solver can recall the answer
K = 4                               # rollouts per solver
TRUNC = Counter()                   # a cut-off answer scores WRONG for length, not for being wrong


def chat(model, prompt, temperature=0.0, max_tokens=1200, tries=6):
    """DeepInfra returns 429 engine_overloaded under load; without backoff one 429 kills
    the whole run 20 minutes in."""
    for i in range(tries):
        try:
            r = client.chat.completions.create(
                model=model, max_tokens=max_tokens, temperature=temperature,
                messages=[{"role": "user", "content": prompt}],
            ).choices[0]
            TRUNC[model] += r.finish_reason == "length"
            return (r.message.content or "").strip()
        except (RateLimitError, APIError):
            if i == tries - 1:
                raise
            time.sleep(2 ** i + random.random())


print(SOURCE[:280])

## 1. The gap

Classic Self-Instruct: ask for training examples, keep what comes back. Score each one with
both solvers — `avg@4`, so the score is a fraction and not a coin flip.

In [ ]:
def parse_examples(raw):
    """The challenger returns JSON; models wrap it in prose often enough to need this."""
    m = re.search(r"\[.*\]", raw, re.S)
    if not m:
        return []
    try:
        out = json.loads(m.group(0))
    except json.JSONDecodeError:
        return []
    return [e for e in out if isinstance(e, dict) and "question" in e and "answer" in e]


# Asking for "answer" with no room to compute gives wrong references, and then BOTH solvers
# score 0.00 — the example dies for a reason that has nothing to do with difficulty.
WORK = """For each example show your arithmetic step by step in "work", then put the exact
final number in "answer". The "answer" MUST follow from "work"."""

VANILLA = f"""Reference material:

<source>
{{source}}
</source>

Write {{n}} question-and-answer training examples answerable from it.
{WORK}
Return ONLY JSON: [{{{{"question": "...", "work": "...", "answer": "..."}}}}]"""

In [ ]:
# Bound the WORKING, not the token budget: at max_tokens=900 the 235B ran past the cap on
# 3 of 4 answers and the judge marked every cut-off one WRONG. Terse -> 0 of 6 truncated.
TERSE = ("Work it out in at most 6 short numbered steps, no prose commentary. "
         "Then end with the final answer on its own last line.")


def solve(model, question):
    return chat(model, f"""Reference material:

<source>
{SOURCE}
</source>

Question: {question}

{TERSE}""", temperature=0.7)


def judge(question, reference, candidate):
    v = chat(JUDGE, f"""Grade a candidate answer against the reference.

Question: {question}
Reference answer: {reference}
Candidate answer: {candidate}

Does the candidate reach the same final answer as the reference? Numeric answers must match
to within 1 credit. Ignore wording, working, and formatting. Reply with exactly one word:
CORRECT or WRONG.""", max_tokens=8)
    return 1.0 if "CORRECT" in v.upper() else 0.0


def score(model, ex):
    with ThreadPoolExecutor(K) as p:
        answers = list(p.map(lambda _: solve(model, ex["question"]), range(K)))
    with ThreadPoolExecutor(K) as p:
        marks = list(p.map(lambda a: judge(ex["question"], ex["answer"], a), answers))
    return sum(marks) / len(marks)


def evaluate(examples):
    with ThreadPoolExecutor(4) as p:
        weak = list(p.map(lambda e: score(WEAK, e), examples))
        strong = list(p.map(lambda e: score(STRONG, e), examples))
    return [dict(e, weak=w, strong=s, gap=s - w)
            for e, w, s in zip(examples, weak, strong)]

The acceptance rule is the whole mechanism. `strong >= 0.65` does double duty: it drops what
is too hard, and it drops examples whose reference answer is simply wrong — the strong model
can't match a broken reference either.

In [ ]:
def accepted(r):
    return r["strong"] >= 0.65 and r["weak"] < 0.5 and r["gap"] >= 0.20


def report(title, rows):
    print(f"\n=== {title} ===")
    for r in rows:
        print(f"  [{'KEEP' if accepted(r) else 'drop'}] weak={r['weak']:.2f} "
              f"strong={r['strong']:.2f} gap={r['gap']:+.2f}  {r['question'][:70]}")
    n = len(rows)
    print(f"  mean weak={sum(r['weak'] for r in rows)/n:.3f}  "
          f"mean strong={sum(r['strong'] for r in rows)/n:.3f}  "
          f"mean gap={sum(r['gap'] for r in rows)/n:+.3f}  "
          f"kept={sum(accepted(r) for r in rows)}/{n}")
    return rows


N = 10
vanilla = report("vanilla self-instruct",
                 evaluate(parse_examples(chat(CHALLENGER, VANILLA.format(source=SOURCE, n=N),
                                              temperature=1.0, max_tokens=4000))))

## 2. The four subagents

Same challenger, same scratchpad, same source. The only thing added is the instruction to
target the gap — so anything that changes is difficulty, not reference quality.

In [ ]:
AGENTIC = f"""Reference material:

<source>
{{source}}
</source>

Write {{n}} question-and-answer training examples answerable ONLY from it.

These train a model, so an example is worthless unless it SEPARATES a strong model from a weak
one. The weak model ({{weak}}) must get it WRONG; the strong model ({{strong}}) must get it
RIGHT. Compose several rules in the right ORDER — discounts before surcharges, rush last, an
eligibility rule that voids a discount. A single table lookup is useless. But if it is so
tangled that even the strong model misses it, it is equally useless.
{WORK}
{{feedback}}
Return ONLY JSON: [{{{{"question": "...", "work": "...", "answer": "..."}}}}]"""

agentic = report("agentic self-instruct (round 1)",
                 evaluate(parse_examples(chat(
                     CHALLENGER,
                     AGENTIC.format(source=SOURCE, n=N, feedback="", weak=WEAK, strong=STRONG),
                     temperature=1.0, max_tokens=4000))))

## 3. Feeding the rejects back

The orchestrator tells the challenger why each example died. Watch this one carefully — it is
the step with no guard on it.

In [ ]:
def feedback_from(rows):
    lines = []
    for r in rows:
        if accepted(r):
            continue
        if r["strong"] < 0.65:
            why = "REJECTED: even the strong model failed (too hard, or your answer is wrong)"
        elif r["weak"] >= 0.5:
            why = "REJECTED: the weak model already solves it (too easy)"
        else:
            why = "REJECTED: gap too narrow"
        lines.append(f'- "{r["question"][:90]}" -> {why} '
                     f"(weak={r['weak']:.2f}, strong={r['strong']:.2f})")
    if not lines:
        return ""
    return ("\nYour last batch was scored. Learn from it:\n" + "\n".join(lines) + "\n")


round2 = report("agentic self-instruct (round 2, fed the rejects)",
                evaluate(parse_examples(chat(
                    CHALLENGER,
                    AGENTIC.format(source=SOURCE, n=N, feedback=feedback_from(agentic),
                                   weak=WEAK, strong=STRONG),
                    temperature=1.0, max_tokens=4000))))

print(f"\ntruncated responses (must be 0 — a cut-off answer scores WRONG): {dict(TRUNC)}")

## Key findings

Weak = Llama-3.1-70B, strong = Qwen3-235B, judge = Qwen2.5-72B, `avg@4`.

- **The gap is the whole story.** Vanilla self-instruct: mean gap `+0.025`, `1/10` kept. Adding
  the difficulty instruction: `+0.350`, and every kept example is one the strong model aces and
  the weak model fails. The paper's baseline moves `0.02 → 0.314`; this run moved `+0.025 → +0.350`.
- **`strong >= 0.65` catches wrong references for free.** No separate check for reference
  correctness — a broken answer fails the strong solver too, so it never reaches the training set.
- **The challenger collapses to one template.** All 10 agentic questions were "A ⟨N⟩ kg ⟨surcharge⟩
  consignment ... with the Anchor account." It found one gap-opening shape and repeated it.
- **Feeding rejects back did not help here** — round 2's gap fell to `+0.225`. Across runs this
  step went up, down, and flat: unguarded, it is a random walk.

## Weaknesses

| Weakness | What happens | Fix |
|---|---|---|
| **The challenger can't do arithmetic in a JSON field** | Asking for `answer` with no scratchpad gave wrong references; both solvers then scored `0.00` and the example died for the wrong reason | A `"work"` field. `strong >= 0.65` catches the survivors — a broken reference fails the strong model too |
| **The strong model truncates, and truncation reads as WRONG** | At `max_tokens=900` the 235B hit the cap on 3 of 4 answers; the judge never saw a final line and marked them WRONG, suppressing every strong score | Bound the working (`at most 6 short numbered steps`) -> `0/0/0` truncated this run. Assert `finish_reason != "length"` |
| **Unguarded feedback is a random walk** | Round 2 gap `+0.350 -> +0.225`. In other runs the same step went `6/8 -> 1/8` and `3/10 -> 6/10` | none here — the paper accepts a mutation only if validation *strictly improves*. This notebook has no such guard |
| **Challenger mode collapse** | All 10 agentic questions used one template ("... with the Anchor account") | Diversity is not in the objective; add a novelty penalty or dedup against kept questions |
| **`avg@4` is coarse, and judge shares a family with the strong solver** | Three negative-gap rows were solver noise, not bad references (each verified by hand) | `avg@8`; judge with a third vendor or parse numerics directly |